# SAC on LunarLanderContinuous-v2

**Soft Actor-Critic (SAC)** is an off-policy maximum-entropy RL algorithm for continuous action spaces. It maximises a trade-off between expected return and entropy:

$$J(\pi) = \sum_t \mathbb{E}_{(s_t,a_t)\sim\rho_\pi}\left[r(s_t,a_t) + \alpha\mathcal{H}(\pi(\cdot|s_t))\right]$$

SAC uses twin Q-networks to mitigate overestimation bias, and can automatically tune the temperature $\alpha$.

> Paper: [Soft Actor-Critic: Off-Policy Maximum Entropy Deep RL](https://arxiv.org/abs/1801.01290)

In this notebook we train SAC on **LunarLanderContinuous-v2** (continuous action space) using `lit-sb3`.


## Setup


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import gymnasium as gym
import numpy as np

from lit_sb3 import RLEnvDataModule, RLTrainer
from lit_sb3.algorithms.sac.sac import SAC

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir ./

## Create the model

Key hyperparameters:

| Parameter         | Description                                                                   |
| ----------------- | ----------------------------------------------------------------------------- |
| `ent_coef`        | Entropy coefficient $\alpha$; `'auto'` learns it automatically                |
| `target_entropy`  | Target entropy for automatic $\alpha$ tuning; `'auto'` = $-\dim(\mathcal{A})$ |
| `tau`             | Polyak soft-update coefficient for target networks                            |
| `learning_starts` | Warm-up steps with random exploration                                         |


In [ ]:
env_name = "LunarLanderContinuous-v3"

In [ ]:
model = SAC(
    "MlpPolicy",
    verbose=1,
    learning_rate=3e-4,
    buffer_size=50_000,
    learning_starts=1_000,
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    gradient_steps=1,
    ent_coef="auto",
    target_entropy="auto",
    policy_kwargs={"net_arch": [256, 256]},
)
print(model.policy_class.__name__)

## Train

Train through `RLTrainer` so the rollout budget and trainer-level settings are explicit.


In [ ]:
datamodule = RLEnvDataModule(train_env=env_name)
trainer = RLTrainer(max_steps=50_000)
trainer.fit(model, datamodule=datamodule)
print(model.policy)

## Evaluate


In [ ]:
eval_results = trainer.test(
    env=gym.make(env_name),
    n_eval_episodes=10,
)
eval_results[0]